# 📄 PDF → PDF con OCR buscable

### Para usarlo:
1. Tocá el botón **▶** de la celda de abajo.
2. Cuando aparezca **Elegir archivos**, subí tu PDF.
3. Al terminar se descarga automáticamente `nombre_OCR.pdf`.

**No tenés que editar código.**  
El PDF original no se modifica: se crea una copia con una capa de texto invisible para poder usar **Ctrl+F**, seleccionar y copiar texto.

In [ ]:
# ============================================================
# 📄 PDF → PDF CON OCR BUSCABLE
# Tocá ▶ y después elegí tu PDF. No hace falta editar nada.
# ============================================================

import os, sys, subprocess, shutil, pathlib, zipfile, time

def run(cmd):
    subprocess.run(cmd, check=True)

print("🔧 Preparando OCR con GPU...")

# Instalar PaddlePaddle GPU + PaddleOCR + utilidades PDF.
run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "paddlepaddle", "paddlepaddle-gpu"])
run([
    sys.executable, "-m", "pip", "install", "-q",
    "paddlepaddle-gpu==3.3.0",
    "-i", "https://www.paddlepaddle.org.cn/packages/stable/cu126/"
])
run([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "paddleocr", "pymupdf", "pillow"
])

import numpy as np
import paddle
import pymupdf
from PIL import Image
from paddleocr import PaddleOCR
from google.colab import files
from IPython.display import display, HTML

# ---------------- GPU ----------------
if (not paddle.is_compiled_with_cuda()) or paddle.device.cuda.device_count() < 1:
    display(HTML("""
    <div style="padding:18px;border:2px solid #d93025;border-radius:12px;font-family:Arial">
      <b>❌ Colab no tiene una GPU activa.</b><br><br>
      Arriba elegí <b>Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU</b>,
      guardá y volvé a tocar ▶.
    </div>
    """))
    raise RuntimeError("GPU no disponible.")

paddle.set_device("gpu:0")
print("✅ GPU lista:", paddle.device.get_device())

# ---------------- OCR ----------------
print("🧠 Cargando PaddleOCR...")
ocr = PaddleOCR(
    lang="es",
    device="gpu:0",
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=True,
)

# Renderizar a esta resolución para OCR.
DPI = 200
MIN_SCORE = 0.35

# Fuente Unicode disponible normalmente en Colab.
FONT_PATH = "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"
if not os.path.exists(FONT_PATH):
    run(["apt-get", "update", "-qq"])
    run(["apt-get", "install", "-y", "-qq", "fonts-dejavu-core"])

font_for_measure = pymupdf.Font(fontfile=FONT_PATH)

def result_dict(res):
    """Normaliza el objeto Result de PaddleOCR a un dict."""
    data = getattr(res, "json", res)
    if callable(data):
        data = data()
    if isinstance(data, dict) and "res" in data:
        data = data["res"]
    return data

def add_hidden_text(page, texts, scores, boxes, img_w, img_h):
    """Agrega texto invisible exactamente sobre las cajas reconocidas."""
    if img_w <= 0 or img_h <= 0:
        return 0

    sx = page.rect.width / float(img_w)
    sy = page.rect.height / float(img_h)

    # Registrar una fuente Unicode en esta página.
    fontname = "dejavu"
    page.insert_font(fontname=fontname, fontfile=FONT_PATH)

    inserted = 0

    for text, score, box in zip(texts, scores, boxes):
        text = str(text).strip()
        if not text or float(score) < MIN_SCORE:
            continue

        x0, y0, x1, y1 = [float(v) for v in box]
        x0 *= sx; x1 *= sx
        y0 *= sy; y1 *= sy

        # Limitar a la página.
        x0 = max(0.0, min(x0, page.rect.width))
        x1 = max(0.0, min(x1, page.rect.width))
        y0 = max(0.0, min(y0, page.rect.height))
        y1 = max(0.0, min(y1, page.rect.height))

        width = max(1.0, x1 - x0)
        height = max(1.0, y1 - y0)

        # Ajustar el tamaño para que la línea invisible quepa en su caja.
        try:
            unit_width = max(font_for_measure.text_length(text, fontsize=1), 0.01)
        except Exception:
            unit_width = max(len(text) * 0.55, 0.01)

        by_height = height * 0.80
        by_width = width / unit_width * 0.97
        fontsize = max(2.5, min(by_height, by_width))

        # Línea base aproximada dentro de la caja detectada.
        baseline = y0 + min(height * 0.82, fontsize * 1.08)

        try:
            page.insert_text(
                pymupdf.Point(x0, baseline),
                text,
                fontname=fontname,
                fontsize=fontsize,
                render_mode=3,   # texto invisible pero seleccionable/buscable
                overlay=True,
            )
            inserted += 1
        except Exception:
            # Un carácter extraño no debe arruinar todo el PDF.
            safe_text = text.encode("utf-8", errors="ignore").decode("utf-8", errors="ignore")
            if safe_text:
                try:
                    page.insert_text(
                        pymupdf.Point(x0, baseline),
                        safe_text,
                        fontname=fontname,
                        fontsize=fontsize,
                        render_mode=3,
                        overlay=True,
                    )
                    inserted += 1
                except Exception:
                    pass

    return inserted

def ocr_pdf(input_path, output_path):
    doc = pymupdf.open(str(input_path))
    total = len(doc)
    total_lines = 0

    for i in range(total):
        page = doc[i]

        # Si el PDF usa rotación de página, la eliminamos sin cambiar su apariencia.
        # Así las coordenadas del OCR coinciden con las coordenadas de inserción.
        if page.rotation:
            page.remove_rotation()

        print(f"📄 {input_path.name}: página {i+1}/{total}")

        pix = page.get_pixmap(
            dpi=DPI,
            colorspace=pymupdf.csRGB,
            alpha=False
        )

        # Pasar la imagen renderizada directamente a PaddleOCR.
        img = Image.frombytes("RGB", (pix.width, pix.height), pix.samples)
        arr = np.asarray(img)

        results = ocr.predict(arr)

        # Para una imagen/página normal se espera un resultado.
        # Igual iteramos por robustez.
        for res in results:
            data = result_dict(res)
            texts = list(data.get("rec_texts", []))
            scores = list(data.get("rec_scores", []))
            boxes = data.get("rec_boxes", [])

            total_lines += add_hidden_text(
                page, texts, scores, boxes, pix.width, pix.height
            )

    # Guardar como un PDF nuevo. El original no se modifica.
    doc.save(
        str(output_path),
        garbage=4,
        deflate=True,
        clean=True
    )
    doc.close()
    return total, total_lines

# ---------------- SUBIR PDF ----------------
display(HTML("""
<div style="padding:16px 18px;border:1px solid #ddd;border-radius:12px;font-family:Arial">
  <b>⬆️ Ahora elegí uno o varios archivos PDF.</b><br>
  El original no se modifica.
</div>
"""))

uploaded = files.upload()

pdfs = []
for filename in uploaded:
    path = pathlib.Path("/content") / filename
    if path.suffix.lower() == ".pdf":
        pdfs.append(path)

if not pdfs:
    raise ValueError("No se seleccionó ningún archivo PDF.")

# ---------------- PROCESAR ----------------
outputs = []

for pdf in pdfs:
    out = pathlib.Path("/content") / f"{pdf.stem}_OCR.pdf"
    pages, lines = ocr_pdf(pdf, out)
    outputs.append(out)
    print(f"✅ {pdf.name}: {pages} páginas, {lines} líneas OCR.")

# ---------------- DESCARGAR ----------------
if len(outputs) == 1:
    display(HTML("""
    <div style="padding:18px;border:2px solid #188038;border-radius:12px;font-family:Arial">
      <b>✅ Listo.</b> El PDF con OCR se va a descargar ahora.
    </div>
    """))
    files.download(str(outputs[0]))
else:
    zip_path = "/content/PDFs_con_OCR.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for p in outputs:
            z.write(p, arcname=p.name)

    display(HTML("""
    <div style="padding:18px;border:2px solid #188038;border-radius:12px;font-family:Arial">
      <b>✅ Listo.</b> Como subiste varios PDFs, se descargan juntos en un ZIP.
    </div>
    """))
    files.download(zip_path)